In [1]:
from pathlib import Path
from PIL import Image

processed_path = Path("../data/images/processed")

valid_extensions = [".jpg", ".jpeg", ".png", ".webp"]

bad_images = []

total_images = 0

for folder in processed_path.iterdir():

    if not folder.is_dir():
        continue

    for image_path in folder.iterdir():

        if image_path.suffix.lower() not in valid_extensions:
            continue

        total_images += 1

        try:
            with Image.open(image_path) as img:
                img.load()

        except Exception as e:
            bad_images.append((image_path, str(e)))


print("Total processed images:", total_images)
print("Images that cannot be opened:", len(bad_images))

for image_path, error in bad_images:
    print(image_path)
    print(error)

In [2]:
import hashlib
from collections import defaultdict

hash_to_files = defaultdict(list)

for folder in processed_path.iterdir():

    if not folder.is_dir():
        continue

    for image_path in folder.iterdir():

        if image_path.suffix.lower() not in valid_extensions:
            continue

        with open(image_path, "rb") as f:
            file_hash = hashlib.md5(f.read()).hexdigest()

        hash_to_files[file_hash].append(image_path)


remaining_duplicates = []

for file_hash, files in hash_to_files.items():

    if len(files) > 1:
        remaining_duplicates.append(files)


print("Remaining duplicate groups:", len(remaining_duplicates))

In [3]:
print("Final processed dataset:\n")

total = 0

for folder in sorted(processed_path.iterdir()):

    if not folder.is_dir():
        continue

    count = sum(
        1
        for file in folder.iterdir()
        if file.suffix.lower() in valid_extensions
    )

    total += count

    print(f"{folder.name}: {count}")

print("\nTotal:", total)

In [4]:
from pathlib import Path

processed_path = Path("../data/images/processed")
preprocessed_path = Path("../data/images/preprocessed")

preprocessed_path.mkdir(parents=True, exist_ok=True)

print("Preprocessed folder:", preprocessed_path)
print("Exists:", preprocessed_path.exists())

In [5]:
for class_folder in sorted(processed_path.iterdir()):

    if class_folder.is_dir():

        output_folder = preprocessed_path / class_folder.name
        output_folder.mkdir(parents=True, exist_ok=True)

        print("Created:", output_folder)

In [6]:
import cv2

print("OpenCV version:", cv2.__version__)

In [7]:
from pathlib import Path
import cv2
import matplotlib.pyplot as plt

# Select one image
sample_image = next(
    processed_path.glob("Black_Soil/*")
)

print("Image:", sample_image)

# Read image
image = cv2.imread(str(sample_image))

# OpenCV loads images as BGR
image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

print("Original shape:", image_rgb.shape)

# Resize
resized = cv2.resize(image_rgb, (224, 224))

print("Resized shape:", resized.shape)

# Display
plt.figure(figsize=(5, 5))
plt.imshow(resized)
plt.title("Resized Black Soil Image")
plt.axis("off")
plt.show()

In [8]:
normalized = resized.astype("float32") / 255.0

print("Minimum pixel value:", normalized.min())
print("Maximum pixel value:", normalized.max())
print("Data type:", normalized.dtype)

In [9]:
plt.figure(figsize=(10, 5))

plt.subplot(1, 2, 1)
plt.imshow(image_rgb)
plt.title("Original")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(resized)
plt.title("Resized 224 × 224")
plt.axis("off")

plt.show()

In [10]:
def preprocess_image(image_path, target_size=(224, 224)):

    image = cv2.imread(str(image_path))

    if image is None:
        raise ValueError(f"Could not read image: {image_path}")

    # BGR → RGB
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    # Resize
    image = cv2.resize(image, target_size)

    # Normalize
    image = image.astype("float32") / 255.0

    return image

In [11]:
sample = preprocess_image(sample_image)

print("Shape:", sample.shape)
print("Minimum:", sample.min())
print("Maximum:", sample.max())
print("Type:", sample.dtype)

In [12]:
import cv2
import matplotlib.pyplot as plt

# Use the same sample image
image = cv2.imread(str(sample_image))

# Convert BGR to RGB for display
image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

# Convert RGB image to HSV
hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)

print("Original shape:", image_rgb.shape)
print("HSV shape:", hsv.shape)

In [13]:
# HSV ranges for a broad soil-like color range
lower = (5, 30, 20)
upper = (40, 255, 255)

mask = cv2.inRange(hsv, lower, upper)

print("Mask created")
print("Mask shape:", mask.shape)

In [14]:
plt.figure(figsize=(10, 5))

plt.subplot(1, 2, 1)
plt.imshow(image_rgb)
plt.title("Original")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(mask, cmap="gray")
plt.title("Initial Background Mask")
plt.axis("off")

plt.show()

In [15]:
masked_image = cv2.bitwise_and(
    image_rgb,
    image_rgb,
    mask=mask
)

plt.figure(figsize=(10, 5))

plt.subplot(1, 2, 1)
plt.imshow(image_rgb)
plt.title("Original")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(masked_image)
plt.title("Masked Image")
plt.axis("off")

plt.show()

In [16]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Read the sample image
image = cv2.imread(str(sample_image))

# Convert BGR to RGB for display
image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

# Create initial mask
mask = np.zeros(image.shape[:2], np.uint8)

# Define a rectangle that contains the main image/soil region
height, width = image.shape[:2]

rect = (
    10,
    int(height * 0.15),
    width - 20,
    int(height * 0.80)
)

# Background and foreground models
bgd_model = np.zeros((1, 65), np.float64)
fgd_model = np.zeros((1, 65), np.float64)

# Apply GrabCut
cv2.grabCut(
    image,
    mask,
    rect,
    bgd_model,
    fgd_model,
    5,
    cv2.GC_INIT_WITH_RECT
)

# Convert mask into binary mask
grabcut_mask = np.where(
    (mask == cv2.GC_FGD) | (mask == cv2.GC_PR_FGD),
    255,
    0
).astype("uint8")

print("GrabCut mask created")
print("Mask shape:", grabcut_mask.shape)

In [17]:
plt.figure(figsize=(10, 5))

plt.subplot(1, 2, 1)
plt.imshow(image_rgb)
plt.title("Original")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(grabcut_mask, cmap="gray")
plt.title("GrabCut Mask")
plt.axis("off")

plt.show()

In [18]:
grabcut_result = cv2.bitwise_and(
    image_rgb,
    image_rgb,
    mask=grabcut_mask
)

plt.figure(figsize=(10, 5))

plt.subplot(1, 2, 1)
plt.imshow(image_rgb)
plt.title("Original")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(grabcut_result)
plt.title("GrabCut Result")
plt.axis("off")

plt.show()

In [19]:
from pathlib import Path
import cv2
import numpy as np
import matplotlib.pyplot as plt

processed_path = Path("../data/images/processed")

valid_extensions = [".jpg", ".jpeg", ".png", ".webp"]

def apply_grabcut(image_path):

    # Read image
    image = cv2.imread(str(image_path))

    if image is None:
        raise ValueError(f"Could not read image: {image_path}")

    height, width = image.shape[:2]

    # Initial mask
    mask = np.zeros((height, width), np.uint8)

    # Rectangle containing the main foreground
    rect = (
        10,
        int(height * 0.10),
        width - 20,
        int(height * 0.85)
    )

    # Models required by GrabCut
    bgd_model = np.zeros((1, 65), np.float64)
    fgd_model = np.zeros((1, 65), np.float64)

    # GrabCut
    cv2.grabCut(
        image,
        mask,
        rect,
        bgd_model,
        fgd_model,
        5,
        cv2.GC_INIT_WITH_RECT
    )

    # Create binary mask
    final_mask = np.where(
        (mask == cv2.GC_FGD) |
        (mask == cv2.GC_PR_FGD),
        255,
        0
    ).astype("uint8")

    # Convert BGR → RGB
    image_rgb = cv2.cvtColor(
        image,
        cv2.COLOR_BGR2RGB
    )

    # Apply mask
    result = cv2.bitwise_and(
        image_rgb,
        image_rgb,
        mask=final_mask
    )

    return image_rgb, final_mask, result

In [20]:
classes = [
    "Alluvial_Soil",
    "Arid_Soil",
    "Black_Soil",
    "Laterite_Soil",
    "Mountain_Soil",
    "Red_Soil",
    "Yellow_Soil"
]

for class_name in classes:

    class_path = processed_path / class_name

    images = [
        p for p in class_path.iterdir()
        if p.suffix.lower() in valid_extensions
    ]

    if len(images) == 0:
        continue

    # Use first image for testing
    image_path = images[0]

    original, mask, result = apply_grabcut(image_path)

    plt.figure(figsize=(12, 4))

    plt.subplot(1, 3, 1)
    plt.imshow(original)
    plt.title(f"{class_name}\nOriginal")
    plt.axis("off")

    plt.subplot(1, 3, 2)
    plt.imshow(mask, cmap="gray")
    plt.title("GrabCut Mask")
    plt.axis("off")

    plt.subplot(1, 3, 3)
    plt.imshow(result)
    plt.title("GrabCut Result")
    plt.axis("off")

    plt.tight_layout()
    plt.show()

In [21]:
import cv2
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt

In [22]:
def preprocess_soil_image(image_path, target_size=(224, 224)):

    # --------------------------------
    # 1. Read image
    # --------------------------------
    image = cv2.imread(str(image_path))

    if image is None:
        raise ValueError(f"Could not read image: {image_path}")

    # --------------------------------
    # 2. Convert BGR → RGB
    # --------------------------------
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    # --------------------------------
    # 3. Color correction using LAB + CLAHE
    # --------------------------------
    lab = cv2.cvtColor(image, cv2.COLOR_RGB2LAB)

    l_channel, a_channel, b_channel = cv2.split(lab)

    clahe = cv2.createCLAHE(
        clipLimit=2.0,
        tileGridSize=(8, 8)
    )

    l_channel = clahe.apply(l_channel)

    lab = cv2.merge(
        [l_channel, a_channel, b_channel]
    )

    image = cv2.cvtColor(
        lab,
        cv2.COLOR_LAB2RGB
    )

    # --------------------------------
    # 4. Aspect-ratio-preserving resize
    # --------------------------------
    target_width, target_height = target_size

    height, width = image.shape[:2]

    scale = min(
        target_width / width,
        target_height / height
    )

    new_width = int(width * scale)
    new_height = int(height * scale)

    resized = cv2.resize(
        image,
        (new_width, new_height),
        interpolation=cv2.INTER_AREA
    )

    # --------------------------------
    # 5. Add padding to reach 224 × 224
    # --------------------------------
    canvas = np.zeros(
        (target_height, target_width, 3),
        dtype=np.uint8
    )

    x_offset = (target_width - new_width) // 2
    y_offset = (target_height - new_height) // 2

    canvas[
        y_offset:y_offset + new_height,
        x_offset:x_offset + new_width
    ] = resized

    # --------------------------------
    # 6. Normalize pixel values
    # --------------------------------
    normalized = canvas.astype(
        np.float32
    ) / 255.0

    return normalized

In [23]:
sample_image = Path(
    "../data/images/processed/Black_Soil/1.jpg"
)

processed_image = preprocess_soil_image(
    sample_image
)

print("Output shape:", processed_image.shape)
print("Data type:", processed_image.dtype)
print("Minimum:", processed_image.min())
print("Maximum:", processed_image.max())

In [24]:
original = cv2.imread(str(sample_image))

original = cv2.cvtColor(
    original,
    cv2.COLOR_BGR2RGB
)

plt.figure(figsize=(10, 5))

plt.subplot(1, 2, 1)
plt.imshow(original)
plt.title("Original")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(processed_image)
plt.title("Final Preprocessed")
plt.axis("off")

plt.show()

In [25]:
def preprocess_and_save_image(image_path, output_path, target_size=(224, 224)):

    # Read image
    image = cv2.imread(str(image_path))

    if image is None:
        raise ValueError(f"Could not read image: {image_path}")

    # BGR → RGB
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    # -----------------------------
    # Color correction
    # -----------------------------
    lab = cv2.cvtColor(image, cv2.COLOR_RGB2LAB)

    l_channel, a_channel, b_channel = cv2.split(lab)

    clahe = cv2.createCLAHE(
        clipLimit=2.0,
        tileGridSize=(8, 8)
    )

    l_channel = clahe.apply(l_channel)

    lab = cv2.merge(
        [l_channel, a_channel, b_channel]
    )

    image = cv2.cvtColor(
        lab,
        cv2.COLOR_LAB2RGB
    )

    # -----------------------------
    # Aspect-ratio-preserving resize
    # -----------------------------
    target_width, target_height = target_size

    height, width = image.shape[:2]

    scale = min(
        target_width / width,
        target_height / height
    )

    new_width = int(width * scale)
    new_height = int(height * scale)

    resized = cv2.resize(
        image,
        (new_width, new_height),
        interpolation=cv2.INTER_AREA
    )

    # -----------------------------
    # Padding
    # -----------------------------
    canvas = np.zeros(
        (target_height, target_width, 3),
        dtype=np.uint8
    )

    x_offset = (target_width - new_width) // 2
    y_offset = (target_height - new_height) // 2

    canvas[
        y_offset:y_offset + new_height,
        x_offset:x_offset + new_width
    ] = resized

    # -----------------------------
    # Save as PNG
    # -----------------------------
    output_path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    # RGB → BGR for OpenCV saving
    canvas_bgr = cv2.cvtColor(
        canvas,
        cv2.COLOR_RGB2BGR
    )

    cv2.imwrite(
        str(output_path),
        canvas_bgr
    )

In [26]:
def preprocess_and_save_image(image_path, output_path, target_size=(224, 224)):

    # Read image
    image = cv2.imread(str(image_path))

    if image is None:
        raise ValueError(f"Could not read image: {image_path}")

    # BGR → RGB
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    # -----------------------------
    # Color correction
    # -----------------------------
    lab = cv2.cvtColor(image, cv2.COLOR_RGB2LAB)

    l_channel, a_channel, b_channel = cv2.split(lab)

    clahe = cv2.createCLAHE(
        clipLimit=2.0,
        tileGridSize=(8, 8)
    )

    l_channel = clahe.apply(l_channel)

    lab = cv2.merge(
        [l_channel, a_channel, b_channel]
    )

    image = cv2.cvtColor(
        lab,
        cv2.COLOR_LAB2RGB
    )

    # -----------------------------
    # Aspect-ratio-preserving resize
    # -----------------------------
    target_width, target_height = target_size

    height, width = image.shape[:2]

    scale = min(
        target_width / width,
        target_height / height
    )

    new_width = int(width * scale)
    new_height = int(height * scale)

    resized = cv2.resize(
        image,
        (new_width, new_height),
        interpolation=cv2.INTER_AREA
    )

    # -----------------------------
    # Padding
    # -----------------------------
    canvas = np.zeros(
        (target_height, target_width, 3),
        dtype=np.uint8
    )

    x_offset = (target_width - new_width) // 2
    y_offset = (target_height - new_height) // 2

    canvas[
        y_offset:y_offset + new_height,
        x_offset:x_offset + new_width
    ] = resized

    # -----------------------------
    # Save as PNG
    # -----------------------------
    output_path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    # RGB → BGR for OpenCV saving
    canvas_bgr = cv2.cvtColor(
        canvas,
        cv2.COLOR_RGB2BGR
    )

    cv2.imwrite(
        str(output_path),
        canvas_bgr
    )

In [27]:
test_input = Path(
    "../data/images/processed/Black_Soil/1.jpg"
)

test_output = Path(
    "../data/images/preprocessed/Black_Soil/1.png"
)

preprocess_and_save_image(
    test_input,
    test_output
)

print("Saved:", test_output)
print("Exists:", test_output.exists())

In [28]:
saved = cv2.imread(str(test_output))

print("Saved image shape:", saved.shape)

In [29]:
saved_rgb = cv2.cvtColor(
    saved,
    cv2.COLOR_BGR2RGB
)

plt.figure(figsize=(5, 5))
plt.imshow(saved_rgb)
plt.title("Saved Preprocessed Image")
plt.axis("off")
plt.show()

In [30]:
# Process the complete cleaned dataset

processed_path = Path("../data/images/processed")
preprocessed_path = Path("../data/images/preprocessed")

valid_extensions = [".jpg", ".jpeg", ".png", ".webp"]

total_processed = 0
failed_images = []

for class_folder in sorted(processed_path.iterdir()):

    if not class_folder.is_dir():
        continue

    class_name = class_folder.name

    output_folder = preprocessed_path / class_name
    output_folder.mkdir(parents=True, exist_ok=True)

    print(f"\nProcessing: {class_name}")

    for image_path in sorted(class_folder.iterdir()):

        if image_path.suffix.lower() not in valid_extensions:
            continue

        try:

            output_path = output_folder / (
                image_path.stem + ".png"
            )

            preprocess_and_save_image(
                image_path,
                output_path
            )

            total_processed += 1

        except Exception as e:

            failed_images.append({
                "file": str(image_path),
                "error": str(e)
            })

    print("Completed:", class_name)


print("\n==============================")
print("PREPROCESSING COMPLETE")
print("==============================")

print("Images processed:", total_processed)
print("Images failed:", len(failed_images))

if failed_images:

    print("\nFailed images:")

    for item in failed_images:
        print(item["file"])
        print(item["error"])

In [31]:
preprocessed_count = 0
bad_preprocessed = []

for class_folder in sorted(preprocessed_path.iterdir()):

    if not class_folder.is_dir():
        continue

    for image_path in class_folder.iterdir():

        if image_path.suffix.lower() != ".png":
            continue

        preprocessed_count += 1

        try:

            image = cv2.imread(str(image_path))

            if image is None:
                raise ValueError("Could not read image")

            if image.shape != (224, 224, 3):
                raise ValueError(
                    f"Unexpected shape: {image.shape}"
                )

        except Exception as e:

            bad_preprocessed.append(
                (image_path, str(e))
            )


print("Preprocessed images:", preprocessed_count)
print("Invalid preprocessed images:", len(bad_preprocessed))

In [32]:
print("Preprocessed class distribution:\n")

total = 0

for class_folder in sorted(preprocessed_path.iterdir()):

    if not class_folder.is_dir():
        continue

    count = sum(
        1
        for file in class_folder.iterdir()
        if file.suffix.lower() == ".png"
    )

    total += count

    print(f"{class_folder.name}: {count}")

print("\nTotal:", total)

In [33]:
import random

for class_name in sorted(
    [p.name for p in preprocessed_path.iterdir() if p.is_dir()]
):

    class_folder = preprocessed_path / class_name

    images = list(class_folder.glob("*.png"))

    selected = random.sample(
        images,
        min(3, len(images))
    )

    fig, axes = plt.subplots(
        1,
        len(selected),
        figsize=(12, 4)
    )

    if len(selected) == 1:
        axes = [axes]

    for ax, image_path in zip(axes, selected):

        image = cv2.imread(str(image_path))

        image = cv2.cvtColor(
            image,
            cv2.COLOR_BGR2RGB
        )

        ax.imshow(image)
        ax.set_title(image_path.name)
        ax.axis("off")

    plt.suptitle(class_name)
    plt.tight_layout()
    plt.show()

In [34]:
def preprocess_and_save_image(image_path, output_path, target_size=(224, 224)):

    # Read image
    image = cv2.imread(str(image_path))

    if image is None:
        raise ValueError(f"Could not read image: {image_path}")

    # BGR → RGB
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    # -----------------------------
    # Color correction
    # -----------------------------
    lab = cv2.cvtColor(image, cv2.COLOR_RGB2LAB)

    l_channel, a_channel, b_channel = cv2.split(lab)

    clahe = cv2.createCLAHE(
        clipLimit=2.0,
        tileGridSize=(8, 8)
    )

    l_channel = clahe.apply(l_channel)

    lab = cv2.merge(
        [l_channel, a_channel, b_channel]
    )

    image = cv2.cvtColor(
        lab,
        cv2.COLOR_LAB2RGB
    )

    # -----------------------------
    # Aspect-ratio-preserving resize
    # -----------------------------
    target_width, target_height = target_size

    height, width = image.shape[:2]

    scale = min(
        target_width / width,
        target_height / height
    )

    new_width = int(width * scale)
    new_height = int(height * scale)

    resized = cv2.resize(
        image,
        (new_width, new_height),
        interpolation=cv2.INTER_AREA
    )

    # -----------------------------
    # Padding
    # -----------------------------
   # Calculate padding
pad_width = target_width - new_width
pad_height = target_height - new_height

top = pad_height // 2
bottom = pad_height - top

left = pad_width // 2
right = pad_width - left

# Reflection padding
canvas = cv2.copyMakeBorder(
    resized,
    top,
    bottom,
    left,
    right,
    cv2.BORDER_REFLECT_101
)

    # -----------------------------
    # Save as PNG
    # -----------------------------
    output_path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    # RGB → BGR for OpenCV saving
    canvas_bgr = cv2.cvtColor(
        canvas,
        cv2.COLOR_RGB2BGR
    )

    cv2.imwrite(
        str(output_path),
        canvas_bgr
    )

In [35]:
def preprocess_and_save_image(image_path, output_path, target_size=(224, 224)):

    # Read image
    image = cv2.imread(str(image_path))

    if image is None:
        raise ValueError(f"Could not read image: {image_path}")

    # BGR → RGB
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    # -----------------------------
    # Color correction
    # -----------------------------
    lab = cv2.cvtColor(image, cv2.COLOR_RGB2LAB)

    l_channel, a_channel, b_channel = cv2.split(lab)

    clahe = cv2.createCLAHE(
        clipLimit=2.0,
        tileGridSize=(8, 8)
    )

    l_channel = clahe.apply(l_channel)

    lab = cv2.merge(
        [l_channel, a_channel, b_channel]
    )

    image = cv2.cvtColor(
        lab,
        cv2.COLOR_LAB2RGB
    )

    # -----------------------------
    # Aspect-ratio-preserving resize
    # -----------------------------
    target_width, target_height = target_size

    height, width = image.shape[:2]

    scale = min(
        target_width / width,
        target_height / height
    )

    new_width = int(width * scale)
    new_height = int(height * scale)

    resized = cv2.resize(
        image,
        (new_width, new_height),
        interpolation=cv2.INTER_AREA
    )

    # -----------------------------
    # Padding
    # -----------------------------
        # Calculate padding
pad_width = target_width - new_width
pad_height = target_height - new_height

top = pad_height // 2
bottom = pad_height - top

left = pad_width // 2
right = pad_width - left

# Reflection padding
canvas = cv2.copyMakeBorder(
    resized,
    top,
    bottom,
    left,
    right,
    cv2.BORDER_REFLECT_101
)

    # -----------------------------
    # Save as PNG
    # -----------------------------
    output_path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    # RGB → BGR for OpenCV saving
    canvas_bgr = cv2.cvtColor(
        canvas,
        cv2.COLOR_RGB2BGR
    )

    cv2.imwrite(
        str(output_path),
        canvas_bgr
    )

In [36]:
def preprocess_and_save_image(image_path, output_path, target_size=(224, 224)):

    # 1. Read image
    image = cv2.imread(str(image_path))

    if image is None:
        raise ValueError(f"Could not read image: {image_path}")

    # 2. Convert BGR to RGB
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    # 3. Color correction using LAB + CLAHE
    lab = cv2.cvtColor(image, cv2.COLOR_RGB2LAB)

    l_channel, a_channel, b_channel = cv2.split(lab)

    clahe = cv2.createCLAHE(
        clipLimit=2.0,
        tileGridSize=(8, 8)
    )

    l_channel = clahe.apply(l_channel)

    lab = cv2.merge(
        [l_channel, a_channel, b_channel]
    )

    image = cv2.cvtColor(
        lab,
        cv2.COLOR_LAB2RGB
    )

    # 4. Aspect-ratio-preserving resize
    target_width, target_height = target_size

    height, width = image.shape[:2]

    scale = min(
        target_width / width,
        target_height / height
    )

    new_width = int(width * scale)
    new_height = int(height * scale)

    resized = cv2.resize(
        image,
        (new_width, new_height),
        interpolation=cv2.INTER_AREA
    )

    # 5. Reflection padding
    pad_width = target_width - new_width
    pad_height = target_height - new_height

    top = pad_height // 2
    bottom = pad_height - top

    left = pad_width // 2
    right = pad_width - left

    canvas = cv2.copyMakeBorder(
        resized,
        top,
        bottom,
        left,
        right,
        cv2.BORDER_REFLECT_101
    )

    # 6. Save as PNG
    output_path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    # RGB → BGR before saving
    canvas_bgr = cv2.cvtColor(
        canvas,
        cv2.COLOR_RGB2BGR
    )

    cv2.imwrite(
        str(output_path),
        canvas_bgr
    )

In [37]:
test_input = Path(
    "../data/images/processed/Black_Soil/1.jpg"
)

test_output = Path(
    "../data/images/preprocessed/Black_Soil/1.png"
)

preprocess_and_save_image(
    test_input,
    test_output
)

print("Saved:", test_output)
print("Exists:", test_output.exists())

In [38]:
test_input = Path(
    "../data/images/processed/Black_Soil/1.jpg"
)

test_output = Path(
    "../data/images/preprocessed/Black_Soil/1.png"
)

preprocess_and_save_image(
    test_input,
    test_output
)

print("Saved:", test_output)
print("Exists:", test_output.exists())

In [39]:
saved = cv2.imread(str(test_output))

print("Saved image shape:", saved.shape)

In [40]:
saved_rgb = cv2.cvtColor(
    saved,
    cv2.COLOR_BGR2RGB
)

plt.figure(figsize=(5, 5))
plt.imshow(saved_rgb)
plt.title("Saved Preprocessed Image")
plt.axis("off")
plt.show()

In [41]:
# Process the complete cleaned dataset

processed_path = Path("../data/images/processed")
preprocessed_path = Path("../data/images/preprocessed")

valid_extensions = [".jpg", ".jpeg", ".png", ".webp"]

total_processed = 0
failed_images = []

for class_folder in sorted(processed_path.iterdir()):

    if not class_folder.is_dir():
        continue

    class_name = class_folder.name

    output_folder = preprocessed_path / class_name
    output_folder.mkdir(parents=True, exist_ok=True)

    print(f"\nProcessing: {class_name}")

    for image_path in sorted(class_folder.iterdir()):

        if image_path.suffix.lower() not in valid_extensions:
            continue

        try:

            output_path = output_folder / (
                image_path.stem + ".png"
            )

            preprocess_and_save_image(
                image_path,
                output_path
            )

            total_processed += 1

        except Exception as e:

            failed_images.append({
                "file": str(image_path),
                "error": str(e)
            })

    print("Completed:", class_name)


print("\n==============================")
print("PREPROCESSING COMPLETE")
print("==============================")

print("Images processed:", total_processed)
print("Images failed:", len(failed_images))

if failed_images:

    print("\nFailed images:")

    for item in failed_images:
        print(item["file"])
        print(item["error"])

In [42]:
def preprocess_and_save_image(image_path, output_path, target_size=(224, 224)):

    # 1. Read image
    image = cv2.imread(str(image_path))

    if image is None:
        raise ValueError(f"Could not read image: {image_path}")

    # 2. Convert BGR → RGB
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    # 3. Color correction using LAB + CLAHE
    lab = cv2.cvtColor(image, cv2.COLOR_RGB2LAB)

    l_channel, a_channel, b_channel = cv2.split(lab)

    clahe = cv2.createCLAHE(
        clipLimit=2.0,
        tileGridSize=(8, 8)
    )

    l_channel = clahe.apply(l_channel)

    lab = cv2.merge(
        [l_channel, a_channel, b_channel]
    )

    image = cv2.cvtColor(
        lab,
        cv2.COLOR_LAB2RGB
    )

    # 4. Direct resize to 224 × 224
    image = cv2.resize(
        image,
        target_size,
        interpolation=cv2.INTER_AREA
    )

    # 5. Convert RGB → BGR for saving
    image_bgr = cv2.cvtColor(
        image,
        cv2.COLOR_RGB2BGR
    )

    # 6. Create output folder
    output_path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    # 7. Save image
    success = cv2.imwrite(
        str(output_path),
        image_bgr
    )

    if not success:
        raise ValueError(
            f"Could not save image: {output_path}"
        )

In [43]:
test_input = Path(
    "../data/images/processed/Black_Soil/1.jpg"
)

test_output = Path(
    "../data/images/preprocessed/Black_Soil/test_final.png"
)

preprocess_and_save_image(
    test_input,
    test_output
)

print("Saved:", test_output)
print("Exists:", test_output.exists())

In [44]:
test_image = cv2.imread(
    str(test_output)
)

print("Shape:", test_image.shape)

In [45]:
test_image_rgb = cv2.cvtColor(
    test_image,
    cv2.COLOR_BGR2RGB
)

plt.figure(figsize=(5, 5))

plt.imshow(test_image_rgb)

plt.title("Final Preprocessed Image")

plt.axis("off")

plt.show()

In [46]:
test_image_rgb = cv2.cvtColor(
    test_image,
    cv2.COLOR_BGR2RGB
)

plt.figure(figsize=(5, 5))

plt.imshow(test_image_rgb)

plt.title("Final Preprocessed Image")

plt.axis("off")

plt.show()

In [47]:
# Process all cleaned images

processed_path = Path("../data/images/processed")
preprocessed_path = Path("../data/images/preprocessed")

valid_extensions = [".jpg", ".jpeg", ".png", ".webp"]

total_processed = 0
failed_images = []

for class_folder in sorted(processed_path.iterdir()):

    if not class_folder.is_dir():
        continue

    class_name = class_folder.name

    output_folder = preprocessed_path / class_name
    output_folder.mkdir(parents=True, exist_ok=True)

    print(f"\nProcessing: {class_name}")

    for image_path in sorted(class_folder.iterdir()):

        if image_path.suffix.lower() not in valid_extensions:
            continue

        try:
            output_path = output_folder / (
                image_path.stem + ".png"
            )

            preprocess_and_save_image(
                image_path,
                output_path
            )

            total_processed += 1

        except Exception as e:

            failed_images.append({
                "file": str(image_path),
                "error": str(e)
            })

    print(f"Completed: {class_name}")


print("\n==============================")
print("PREPROCESSING COMPLETE")
print("==============================")
print("Images processed:", total_processed)
print("Images failed:", len(failed_images))

In [48]:
preprocessed_count = 0
invalid_images = []

for class_folder in sorted(preprocessed_path.iterdir()):

    if not class_folder.is_dir():
        continue

    for image_path in class_folder.glob("*.png"):

        preprocessed_count += 1

        image = cv2.imread(str(image_path))

        if image is None:
            invalid_images.append(
                str(image_path)
            )

        elif image.shape != (224, 224, 3):
            invalid_images.append(
                f"{image_path} → {image.shape}"
            )


print("Preprocessed images:", preprocessed_count)
print("Invalid images:", len(invalid_images))

if invalid_images:
    print("\nProblems:")
    for item in invalid_images:
        print(item)

In [49]:
preprocessed_count = 0
invalid_images = []

for class_folder in sorted(preprocessed_path.iterdir()):

    if not class_folder.is_dir():
        continue

    for image_path in class_folder.glob("*.png"):

        preprocessed_count += 1

        image = cv2.imread(str(image_path))

        if image is None:
            invalid_images.append(
                str(image_path)
            )

        elif image.shape != (224, 224, 3):
            invalid_images.append(
                f"{image_path} → {image.shape}"
            )


print("Preprocessed images:", preprocessed_count)
print("Invalid images:", len(invalid_images))

if invalid_images:
    print("\nProblems:")
    for item in invalid_images:
        print(item)

In [50]:
print("PREPROCESSED CLASS COUNTS")
print("=========================")

total = 0

for class_folder in sorted(preprocessed_path.iterdir()):

    if not class_folder.is_dir():
        continue

    count = len(list(class_folder.glob("*.png")))

    total += count

    print(f"{class_folder.name}: {count}")

print("=========================")
print("Total:", total)

In [51]:
print("PREPROCESSED CLASS COUNTS")
print("=========================")

total = 0

for class_folder in sorted(preprocessed_path.iterdir()):

    if not class_folder.is_dir():
        continue

    count = len(list(class_folder.glob("*.png")))

    total += count

    print(f"{class_folder.name}: {count}")

print("=========================")
print("Total:", total)

In [52]:
# Compare original cleaned Black Soil files
# with preprocessed Black Soil files

original_black = Path(
    "../data/images/processed/Black_Soil"
)

preprocessed_black = Path(
    "../data/images/preprocessed/Black_Soil"
)

original_names = {
    file.stem
    for file in original_black.iterdir()
    if file.suffix.lower() in [".jpg", ".jpeg", ".png", ".webp"]
}

preprocessed_names = {
    file.stem
    for file in preprocessed_black.glob("*.png")
}

extra_files = preprocessed_names - original_names

print("Extra preprocessed files:")

for name in sorted(extra_files):
    print(name)

print("\nNumber of extra files:", len(extra_files))

In [53]:
extra_file = preprocessed_black / "test_final.png"

if extra_file.exists():
    extra_file.unlink()
    print("Deleted:", extra_file)
else:
    print("File not found")

In [54]:
preprocessed_count = 0
invalid_images = []

for class_folder in sorted(preprocessed_path.iterdir()):

    if not class_folder.is_dir():
        continue

    for image_path in class_folder.glob("*.png"):

        preprocessed_count += 1

        image = cv2.imread(str(image_path))

        if image is None:
            invalid_images.append(str(image_path))

        elif image.shape != (224, 224, 3):
            invalid_images.append(
                f"{image_path} → {image.shape}"
            )

print("Preprocessed images:", preprocessed_count)
print("Invalid images:", len(invalid_images))

In [55]:
extra_file = preprocessed_black / "test_final.png"

if extra_file.exists():
    extra_file.unlink()
    print("Deleted:", extra_file)
else:
    print("File not found")

In [56]:
from sklearn.model_selection import train_test_split
import shutil

split_path = Path("../data/images/split")

classes = sorted([
    folder.name
    for folder in preprocessed_path.iterdir()
    if folder.is_dir()
])

for split in ["train", "validation", "test"]:

    for class_name in classes:

        folder = split_path / split / class_name
        folder.mkdir(
            parents=True,
            exist_ok=True
        )

print("Train, validation and test folders created.")

In [57]:
for class_name in classes:

    class_folder = preprocessed_path / class_name

    images = sorted(
        class_folder.glob("*.png")
    )

    # 70% training, 30% temporary
    train_images, temp_images = train_test_split(
        images,
        test_size=0.30,
        random_state=42
    )

    # Split temporary 50/50
    # → 15% validation
    # → 15% test
    validation_images, test_images = train_test_split(
        temp_images,
        test_size=0.50,
        random_state=42
    )

    # Copy training images
    for image in train_images:

        destination = (
            split_path /
            "train" /
            class_name /
            image.name
        )

        shutil.copy2(
            image,
            destination
        )

    # Copy validation images
    for image in validation_images:

        destination = (
            split_path /
            "validation" /
            class_name /
            image.name
        )

        shutil.copy2(
            image,
            destination
        )

    # Copy test images
    for image in test_images:

        destination = (
            split_path /
            "test" /
            class_name /
            image.name
        )

        shutil.copy2(
            image,
            destination
        )

    print(
        f"{class_name}: "
        f"{len(train_images)} train | "
        f"{len(validation_images)} validation | "
        f"{len(test_images)} test"
    )

In [58]:
for split in ["train", "validation", "test"]:

    print("\n" + split.upper())
    print("-" * 30)

    total = 0

    for class_name in classes:

        folder = (
            split_path /
            split /
            class_name
        )

        count = len(
            list(folder.glob("*.png"))
        )

        total += count

        print(
            f"{class_name}: {count}"
        )

    print("Total:", total)

In [59]:
train_files = set()

for class_name in classes:

    folder = split_path / "train" / class_name

    for image in folder.glob("*.png"):
        train_files.add(image.name)


validation_files = set()

for class_name in classes:

    folder = split_path / "validation" / class_name

    for image in folder.glob("*.png"):
        validation_files.add(image.name)


test_files = set()

for class_name in classes:

    folder = split_path / "test" / class_name

    for image in folder.glob("*.png"):
        test_files.add(image.name)


print("Train ∩ Validation:", len(train_files & validation_files))
print("Train ∩ Test:", len(train_files & test_files))
print("Validation ∩ Test:", len(validation_files & test_files))

In [60]:
import hashlib

def get_file_hash(file_path):

    sha256 = hashlib.sha256()

    with open(file_path, "rb") as f:

        while True:

            data = f.read(8192)

            if not data:
                break

            sha256.update(data)

    return sha256.hexdigest()


def get_hashes(split_name):

    hashes = {}

    split_folder = split_path / split_name

    for class_folder in split_folder.iterdir():

        if not class_folder.is_dir():
            continue

        for image_path in class_folder.glob("*.png"):

            file_hash = get_file_hash(image_path)

            hashes[file_hash] = image_path

    return hashes

In [61]:
train_hashes = get_hashes("train")
validation_hashes = get_hashes("validation")
test_hashes = get_hashes("test")

train_validation = set(train_hashes) & set(validation_hashes)
train_test = set(train_hashes) & set(test_hashes)
validation_test = set(validation_hashes) & set(test_hashes)

print("Actual image duplicates between splits")
print("----------------------------------------")

print(
    "Train ∩ Validation:",
    len(train_validation)
)

print(
    "Train ∩ Test:",
    len(train_test)
)

print(
    "Validation ∩ Test:",
    len(validation_test)
)

In [62]:
print("Train:", len(train_hashes))
print("Validation:", len(validation_hashes))
print("Test:", len(test_hashes))

print(
    "Total:",
    len(train_hashes)
    + len(validation_hashes)
    + len(test_hashes)
)

In [63]:
import tensorflow as tf

print("TensorFlow version:", tf.__version__)

In [64]:
import sys

print("Python executable:")
print(sys.executable)

print("\nPython version:")
print(sys.version)

In [65]:
get_ipython().run_line_magic('pip', 'install tensorflow')

In [66]:
import sys
import platform

print("Python version:", sys.version)
print("Python executable:", sys.executable)
print("Platform:", platform.platform())

In [67]:
get_ipython().system('python --version')

In [68]:
get_ipython().system('python --version')

In [69]:
get_ipython().system('python --version')

In [70]:
get_ipython().system('python --version')